# Multi-Agent Student Task Coordinator
An A2A-inspired prototype using Planner, Research, and Writer agents with Gemini and Gradio.

In [ ]:
!pip install -q gradio pandas requests

In [ ]:
import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")
print("API key saved for this Colab session.")

In [ ]:
import requests

MODEL = "gemini-3.6-flash"
API_URL = "https://generativelanguage.googleapis.com/v1beta/interactions"

def find_text(data):
    collected = []
    if isinstance(data, dict):
        for key, value in data.items():
            if key == "text" and isinstance(value, str):
                collected.append(value)
            else:
                collected.extend(find_text(value))
    elif isinstance(data, list):
        for item in data:
            collected.extend(find_text(item))
    return collected

def call_gemini(prompt):
    headers = {"Content-Type": "application/json", "x-goog-api-key": os.environ["GEMINI_API_KEY"]}
    response = requests.post(API_URL, headers=headers, json={"model": MODEL, "input": prompt}, timeout=90)
    response.raise_for_status()
    text = find_text(response.json())
    if not text:
        raise ValueError("No readable model response was returned.")
    return "\n".join(text)

In [ ]:
import uuid
from datetime import datetime, timezone

AGENT_CARDS = {
    "planner": {"name": "Planner Agent", "skills": ["task planning", "task decomposition"]},
    "researcher": {"name": "Research Agent", "skills": ["topic research", "summarization"]},
    "writer": {"name": "Writer Agent", "skills": ["writing", "formatting"]}
}

def create_message(sender, receiver, instruction):
    return {"task_id": str(uuid.uuid4()), "from_agent": sender, "to_agent": receiver, "instruction": instruction, "status": "submitted", "created_at": datetime.now(timezone.utc).isoformat()}

In [ ]:
def planner_agent(request):
    return call_gemini(f"You are the Planner Agent. Break this student request into 3-5 practical steps. Be concise.\n\nRequest: {request}")

def research_agent(request, plan):
    return call_gemini(f"You are the Research Agent. Develop concise, useful information for the request and plan. Do not invent sources or statistics.\n\nRequest: {request}\n\nPlan: {plan}")

def writer_agent(request, plan, research):
    return call_gemini(f"You are the Writer Agent. Produce a clear, organized, student-friendly final response.\n\nRequest: {request}\n\nPlan: {plan}\n\nResearch: {research}")

In [ ]:
def run_multi_agent_system(user_request):
    log = []
    jobs = [("user", "planner", user_request)]
    planner_message = create_message(*jobs[0]); planner_message["status"] = "working"
    plan = planner_agent(user_request); planner_message["status"] = "completed"; log.append(planner_message)
    research_message = create_message("planner", "researcher", "Research the plan"); research_message["status"] = "working"
    research = research_agent(user_request, plan); research_message["status"] = "completed"; log.append(research_message)
    writer_message = create_message("researcher", "writer", "Create the final response"); writer_message["status"] = "working"
    final = writer_agent(user_request, plan, research); writer_message["status"] = "completed"; log.append(writer_message)
    return {"status": "completed", "plan": plan, "research": research, "final_answer": final, "interaction_log": log}

In [ ]:
import gradio as gr

def process_request(request):
    if not request or not request.strip():
        return "Please enter a request.", "", "", "No task submitted."
    try:
        result = run_multi_agent_system(request)
        log = "\n\n".join(f"{m['from_agent']} -> {m['to_agent']} | {m['status']} | {m['task_id']}" for m in result['interaction_log'])
        return result['plan'], result['research'], result['final_answer'], log
    except Exception as error:
        return "Request failed.", "", "", f"Error: {error}"

with gr.Blocks(title="Multi-Agent Student Coordinator") as demo:
    gr.Markdown("# Multi-Agent Student Task Coordinator\nPlanner, Research, and Writer agents collaborate on academic requests.")
    user_input = gr.Textbox(label="Student Request", lines=4, placeholder="Create a four-week Python study plan.")
    run_button = gr.Button("Run Multi-Agent System", variant="primary")
    with gr.Tab("Planner Agent"): plan_output = gr.Markdown()
    with gr.Tab("Research Agent"): research_output = gr.Markdown()
    with gr.Tab("Writer Agent"): final_output = gr.Markdown()
    with gr.Tab("Communication Log"): log_output = gr.Textbox(lines=10)
    run_button.click(process_request, user_input, [plan_output, research_output, final_output, log_output])

demo.launch(share=True, debug=False)